# Week 59, Day 2: Generators

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w059d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def countdown(n):
    print("starting")
    while n > 0:
        yield n
        n -= 1
    print("finished")

c = countdown(3)
print("made it")
print(next(c))
print(next(c))
print(list(c))

In [ ]:
def evens(limit):
    n = 0
    while n <= limit:
        yield n
        n += 2

g = evens(8)
print(type(g).__name__, iter(g) is g)
print(list(g), list(g), list(evens(4)), sum(evens(100)))

In [ ]:
def chunks(iterable, n):
    group = []
    for item in iterable:
        group.append(item)
        if len(group) == n:
            yield tuple(group)
            group = []
    if group:
        yield tuple(group)

def window(iterable, n):
    from collections import deque
    recent = deque(maxlen=n)
    for item in iterable:
        recent.append(item)
        if len(recent) == n:
            yield tuple(recent)

print(list(chunks("abcdefg", 3)), list(window([1, 2, 3, 4], 2)))

In [ ]:
class Playlist:
    def __init__(self, songs):
        self._songs = list(songs)

    def __iter__(self):
        for song in self._songs:
            yield song.title()

p = Playlist(["intro", "the long walk", "outro"])
print(list(p), list(p))

In [ ]:
import sys
from itertools import islice

def naturals():
    n = 1
    while True:
        yield n
        n += 1

def squares_of(numbers):
    for n in numbers:
        yield n * n

print(list(islice(squares_of(naturals()), 5)))
print(sys.getsizeof(squares_of(range(10 ** 9))), sys.getsizeof(list(range(10 ** 5))))

In [ ]:
def numbered(lines):
    print("open")
    try:
        for i, line in enumerate(lines, 1):
            yield f"{i}: {line}"
    finally:
        print("closed")

g = numbered(["alpha", "beta", "gamma"])
print(next(g))
g.close()                      # stop early: the finally block still runs
print(list(numbered(["x"])))

In [ ]:
def in_order(tree):
    """tree: [key, left, right] lists, None for empty."""
    stack, node = [], tree
    while stack or node:
        while node:
            stack.append(node)
            node = node[1]
        node = stack.pop()
        yield node[0]
        node = node[2]

tree = [5, [3, [1, None, None], [4, None, None]], [8, None, [9, None, None]]]
walk = in_order(tree)
print(next(walk), next(walk), list(walk))
print(next(x for x in in_order(tree) if x > 4))      # stops as soon as it finds one

> **Key idea.** A function containing yield is a generator function: calling it returns a generator without running the body. Each next() runs to the following yield and pauses there, keeping every local variable; finishing or return raises StopIteration. Generators are one-shot iterators, so call the function again to loop again, or make __iter__ a generator method for a reusable iterable. They're lazy: endless sequences, tiny memory, values as soon as they're ready. Put cleanup in try and finally, which runs even when the consumer stops early. Walking structures with a generator lets the consumer stop the walk the moment it has what it needs.

### Exercise: First generators

Write generator functions `fib()` (0, 1, 1, 2, 3, 5, ... for ever), `collatz(n)` (n, then n/2 if even or 3n + 1 if odd, ..., ending with 1; whole numbers), and `repeat_each(iterable, k)` (each item k times in a row). Each must use `yield`.

In [ ]:
def fib():
    ___

def collatz(n):
    ___

def repeat_each(iterable, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, types\ndef test_basics():\n    \"yield\"\n    assert list(itertools.islice(fib(), 10)) == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]\n    assert list(collatz(6)) == [6, 3, 10, 5, 16, 8, 4, 2, 1] and list(collatz(1)) == [1]\n    assert list(repeat_each(\"ab\", 3)) == [\"a\", \"a\", \"a\", \"b\", \"b\", \"b\"] and list(repeat_each([], 2)) == []\n    assert isinstance(fib(), types.GeneratorType) and isinstance(collatz(3), types.GeneratorType)\n    assert isinstance(repeat_each(\"x\", 1), types.GeneratorType)\n    assert list(itertools.islice(repeat_each(itertools.count(), 2), 5)) == [0, 0, 1, 1, 2]")

### Exercise: Reading records lazily

A settings file has lines like `name = value`; blank lines and lines starting with `#` are ignored. Write `settings(lines)`: a generator of `(name, value)` pairs with surrounding spaces stripped, raising `ValueError(f"line {n}: no '='")` (n counting from 1) at a bad line **only when it's reached**. Then `first_setting(lines, name)`: the value of the first matching name, stopping as soon as it's found (so later bad lines don't matter), or `None`.

In [ ]:
def settings(lines):
    ___

def first_setting(lines, name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import types\ndef test_records():\n    \"lazy parsing\"\n    lines = [\"# comment\", \"\", \"colour = blue\", \"  size=10 \", \"colour = red\"]\n    assert list(settings(lines)) == [(\"colour\", \"blue\"), (\"size\", \"10\"), (\"colour\", \"red\")]\n    assert isinstance(settings([]), types.GeneratorType)\n    bad = [\"a = 1\", \"oops\", \"b = 2\"]\n    g = settings(bad)\n    assert next(g) == (\"a\", \"1\")\n    try:\n        next(g)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"line 2: no '='\"\n    assert first_setting(bad, \"a\") == \"1\" and first_setting(lines, \"size\") == \"10\" and first_setting(lines, \"font\") is None\n    def endless():\n        yield \"x = 1\"\n        while True:\n            yield \"junk\"\n    assert first_setting(endless(), \"x\") == \"1\"")

### Exercise: Classes into generators

Rewrite yesterday's classes as generators: `peek_pairs(iterable)`: each item with the one after it, as `(item, next_item)`, with `None` after the last; and `interleave(*iterables)`: one item from each in turn, skipping exhausted ones (a deque of iterators, as yesterday, but with `yield`). Both lazy, both working on endless inputs.

In [ ]:
from collections import deque

def peek_pairs(iterable):
    ___

def interleave(*iterables):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef test_rewrite():\n    \"generators\"\n    assert list(peek_pairs(\"abc\")) == [(\"a\", \"b\"), (\"b\", \"c\"), (\"c\", None)] and list(peek_pairs([])) == []\n    assert list(interleave(\"abc\", \"1\", \"xy\")) == [\"a\", \"1\", \"x\", \"b\", \"y\", \"c\"] and list(interleave()) == []\n    assert list(itertools.islice(peek_pairs(itertools.count()), 3)) == [(0, 1), (1, 2), (2, 3)]\n    assert list(itertools.islice(interleave(itertools.count(), \"ab\"), 6)) == [0, \"a\", 1, \"b\", 2, 3]\n    assert \"class \" not in source and \"yield\" in source")

### Exercise: Primes for ever

Write `primes()`: an endless generator of primes using an **incremental sieve**: a dict maps each upcoming composite number to the primes that will reach it. For each n: if n isn't in the dict it's prime, so yield it and record n² as reached by n; otherwise move each prime p recorded at n on to n + p. (This version is described in Melissa O'Neill's 2009 paper "The Genuine Sieve of Eratosthenes".) Then `nth_prime(k)` (k from 1) using it.

In [ ]:
def primes():
    ___

def nth_prime(k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef test_primes():\n    \"an incremental sieve\"\n    assert list(itertools.islice(primes(), 10)) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]\n    assert nth_prime(1) == 2 and nth_prime(100) == 541 and nth_prime(5000) == 48611\n    small = set(itertools.takewhile(lambda p: p < 2000, primes()))\n    assert small == {n for n in range(2, 2000) if all(n % d for d in range(2, int(n ** 0.5) + 1))}")

### Exercise: Walking a folder tree

Folders are dicts: a key mapping to a dict is a folder, to a number is a file of that size. Write `files(tree, prefix="")`: a generator of `(path, size)` for every file, paths like `"docs/notes.txt"`, visiting everything at each level in sorted name order, depth first (a folder's contents come straight after it, before its later siblings). Use an explicit stack (no recursion). Then `largest_over(tree, limit)`: the first file (in that order) bigger than limit, as `(path, size)`, or `None`, stopping the walk as soon as it's found.

In [ ]:
def files(tree, prefix=""):
    ___

def largest_over(tree, limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_walk():\n    \"a lazy tree walk\"\n    tree = {\"docs\": {\"b.txt\": 5, \"a.txt\": 3, \"old\": {\"x.txt\": 50}}, \"photo.jpg\": 900, \"z\": {}}\n    assert list(files(tree)) == [(\"docs/a.txt\", 3), (\"docs/b.txt\", 5), (\"docs/old/x.txt\", 50), (\"photo.jpg\", 900)]\n    assert largest_over(tree, 10) == (\"docs/old/x.txt\", 50) and largest_over(tree, 1000) is None\n    assert list(files({})) == []\n    deep = {}\n    node = deep\n    for i in range(3000):\n        node[\"d\"] = {}\n        node = node[\"d\"]\n    node[\"f\"] = 1\n    assert next(files(deep))[1] == 1")

### Bug hunt: return, not yield

`return` ends the function at the first value, so this "generator" returns a single number and `list` of it fails. Use `yield` so it prints `[1, 4, 9]`.

In [ ]:
def squares(n):
    for i in range(1, n + 1):
        return i * i

try:
    print(list(squares(3)))
except TypeError as e:
    print("TypeError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"yield\"\n    expect_output(\"[1, 4, 9]\")")

### Bug hunt: The generator object itself

Calling a generator function only makes a generator: this prints something like `<generator object evens at 0x...>` instead of the numbers. Turn it into a list (or loop over it) so it prints `[0, 2, 4]`.

In [ ]:
def evens(n):
    for i in range(0, n, 2):
        yield i

print(evens(6))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"consume it\"\n    expect_output(\"[0, 2, 4]\")")

### Bug hunt: Used up already

A generator is one-shot: the `sum` uses it up, so the `max` afterwards raises `ValueError` (an empty sequence). Call the function again for a fresh generator so it prints `14 9`.

In [ ]:
def readings():
    for x in (3, 9, 2):
        yield x

data = readings()
total = sum(data)
try:
    print(total, max(data))
except ValueError:
    print(total, "nothing left")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a fresh generator\"\n    expect_output(\"14 9\")")

### Puzzle: Chunks as a generator

Arrange the lines of a generator yielding tuples of n items (the last may be shorter). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
def gen():
    yield 1
    yield 2
    return 3

p = ___   # list(gen())
q = ___   # g = gen(); next(g); next(g); next(g, "end")
r = ___   # True or False: calling gen() runs its body up to the first yield
s = ___   # sum(x for x in gen())
t = ___   # g = gen(); list(g) + list(g)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2] and q == \"end\" and r is False, \"p, q and r\"\n    assert s == 3 and t == [1, 2], \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: following pages lazily

A web service returns results a page at a time: `fetch(token)` returns `(items, next_token)`, where `next_token` is `None` after the last page, and the first page uses the token `None`. Fetching is slow, so a program should fetch only the pages it actually needs. Write generators:

- `all_items(fetch)`: every item from every page, in order, fetching each page only when the previous one's items have all been taken.
- `take(n, iterable)`: the first n items as a list (it must not take an (n + 1)-th item from the source).
- `pages_needed(fetch, n)`: how many fetches `take(n, all_items(fetch))` makes, by wrapping `fetch` in a function that counts calls.
- `search(fetch, pred)`: the first item for which `pred` is True, fetching no more pages than needed; `None` if none.

In [ ]:
def all_items(fetch):
    ___

def take(n, iterable):
    ___

def pages_needed(fetch, n):
    ___

def search(fetch, pred):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import types\ndef make(pages):\n    calls = []\n    def fetch(token):\n        calls.append(token)\n        i = 0 if token is None else token\n        nxt = i + 1 if i + 1 < len(pages) else None\n        return pages[i], nxt\n    return fetch, calls\ndef test_pages():\n    \"lazy paging\"\n    pages = [[1, 2, 3], [4, 5], [], [6, 7, 8], [9]]\n    fetch, calls = make(pages)\n    assert isinstance(all_items(fetch), types.GeneratorType) and calls == []\n    assert list(all_items(fetch)) == [1, 2, 3, 4, 5, 6, 7, 8, 9] and len(calls) == 5\n    fetch, calls = make(pages)\n    assert take(4, all_items(fetch)) == [1, 2, 3, 4] and calls == [None, 1]\n    assert take(0, all_items(make(pages)[0])) == [] and take(99, all_items(make(pages)[0])) == list(range(1, 10))\n    assert pages_needed(make(pages)[0], 3) == 1 and pages_needed(make(pages)[0], 6) == 4 and pages_needed(make(pages)[0], 0) == 0\n    fetch, calls = make(pages)\n    assert search(fetch, lambda x: x > 4) == 5 and len(calls) == 2\n    assert search(make(pages)[0], lambda x: x > 100) is None\n    def endless(token):\n        i = token or 0\n        return [i * 10 + k for k in range(10)], i + 1\n    assert take(25, all_items(endless))[-1] == 24 and search(endless, lambda x: x % 97 == 96) == 96")

## Warm-up

Calling `countdown(3)` printed nothing: the body didn't start. The first `next` ran it up to the first `yield`; the second resumed just after that `yield`; `list` ran it to the end, which printed "finished" and raised `StopIteration`.

## yield

A function with `yield` anywhere in its body is a **generator function**. Calling it returns a **generator object** without running any of the body. The generator is an **iterator** (it has `__iter__` returning itself and `__next__`):

1. `next(gen)` runs the body until the next `yield value`, then **pauses** and returns the value. Local variables, the position in loops, everything is kept.
2. The next `next(gen)` resumes just after that `yield`.
3. When the body finishes (falls off the end or runs `return`), the generator raises `StopIteration`, and it stays finished.

Because generators are iterators, everything that loops works with them: `for`, `list`, `sum`, `zip`, unpacking. And like every iterator, a generator is **one-shot**: to loop again, call the function again for a fresh generator.

## Classes become functions

Yesterday's `Chunks` class needed `__init__`, `__iter__`, `__next__`, and careful handling of `StopIteration`. As a generator, the state lives in ordinary local variables and the loop does the work:

The same works for an **iterable** class: make `__iter__` a generator method, and every loop over the object gets a fresh generator, so the object can be looped over again and again.

## Lazy and endless

A generator works out each value only when it's asked for, so:

- It can represent an **endless** sequence (`while True: yield ...`); the consumer decides how much to take, with `break`, `itertools.islice` or `zip` against something finite.
- It needs only the memory for its current state, however many values it produces: a generator over a billion numbers is a few hundred bytes, where the list would be gigabytes.
- Work is **spread out**: the first value arrives immediately rather than after the whole list is built, which matters for streams and slow sources.

## Finishing and cleaning up

`return` inside a generator **ends** it (the value given to `return`, if any, becomes the `StopIteration`'s `value`, which Thursday's `yield from` uses). A generator that holds a resource (an open file, a connection) should release it even if the consumer stops early. Put the cleanup in `try` and `finally`: if a generator is abandoned, Python calls its `close()` method, which raises `GeneratorExit` at the paused `yield`, so the `finally` block runs. (`with` inside a generator works the same way, for the same reason; Week 61 covers context managers.)

## Walking structures

Generators make walking trees and graphs read like the definition. An in-order walk of a binary tree, with an explicit stack so tall trees don't hit the recursion limit:

The last line shows the real power: the search **stops** the walk as soon as it has its answer; nothing after 5 is visited.

## Practice

## Quiz

## Daily challenge

## Before you go

Take one iterator class you wrote in an earlier week (Week 36 or Phase 4) and rewrite it as a generator; count the lines saved. Tomorrow: **generator expressions**, generators in a single line.